<a href="https://colab.research.google.com/github/gokulmahatme8-web/CVD-CaseStudyML/blob/main/Logistic_Regression_CVD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, roc_auc_score)
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 88)   # keep wide tables readable

print("Libraries imported successfully")

Libraries imported successfully


In [10]:
# Load the raw cardiovascular dataset
df = pd.read_csv("cardio_raw.csv")

# Display the first five records
df.head()

,PatientID,Age,Gender,Height,Weight,SystolicBP,DiastolicBP,Cholesterol,Glucose,Smoking,Alcohol,PhysicalActivity,CardioDisease
0,268,61,1,157.0,55.3,179.0,108.0,1.0,1,0,0,1,1
1,143,53,2,167.0,56.6,90.0,71.0,2.0,1,0,1,1,0
2,838,46,1,165.0,69.6,117.0,78.0,1.0,1,0,0,1,0
3,176,44,1,171.0,48.1,150.0,91.0,1.0,3,0,1,1,0
4,482,53,2,165.0,78.4,138.0,93.0,2.0,1,0,0,1,1


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Shape, data types, missing values and duplicates (raw data)
print("Shape of dataset:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Statistical summary of the raw data
df[["Age", "Height", "Weight", "SystolicBP", "DiastolicBP"]].describe().round(1)

In [ ]:
# Step 1: remove duplicate rows and the PatientID column
before = len(df)
df = df.drop_duplicates().drop(columns=["PatientID"])
print("Duplicate rows removed:", before - len(df))
print("Rows remaining        :", len(df))

In [ ]:
# Step 2: replace impossible values with NaN (data-entry errors)
n0 = df.isnull().sum().sum()
df.loc[~df["SystolicBP"].between(80, 250), "SystolicBP"] = np.nan
df.loc[~df["DiastolicBP"].between(40, 150), "DiastolicBP"] = np.nan
df.loc[~df["Height"].between(120, 220), "Height"] = np.nan
df.loc[~df["Weight"].between(30, 180), "Weight"] = np.nan
print("Invalid values flagged as missing:", df.isnull().sum().sum() - n0)

# Step 3: fill missing values (median for numbers, mode for categories)
for col in ["Height", "Weight", "SystolicBP", "DiastolicBP"]:
    df[col] = df[col].fillna(df[col].median())
df["Cholesterol"] = df["Cholesterol"].fillna(df["Cholesterol"].mode()[0])
print("Missing values left   :", df.isnull().sum().sum())

In [ ]:
# Step 4: remove rows where systolic BP is not higher than diastolic BP
bad = (df["SystolicBP"] <= df["DiastolicBP"]).sum()
df = df[df["SystolicBP"] > df["DiastolicBP"]].reset_index(drop=True)
print("Rows removed (systolic <= diastolic):", bad)

# Step 5: create a new feature - Body Mass Index (BMI)
df["BMI"] = (df["Weight"] / (df["Height"] / 100) ** 2).round(1)
df["Cholesterol"] = df["Cholesterol"].astype(int)
df.to_csv("cardio_cleaned.csv", index=False)
print("Cleaned dataset shape:", df.shape)
df.head()

In [ ]:
# Statistical summary after cleaning
df[["Age", "Height", "Weight", "SystolicBP", "DiastolicBP", "BMI"]].describe().round(1)

In [ ]:
# Bar chart: patients with and without cardiovascular disease
counts = df["CardioDisease"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
bars = plt.bar(["No Disease (0)", "Disease (1)"], counts.values, color=["#4C78A8", "#E45756"])
plt.bar_label(bars)
plt.title("Number of Patients by Cardiovascular Disease")
plt.ylabel("Number of Patients")
plt.show()
print(counts)

In [ ]:
# Bar chart: disease rate for each cholesterol level
rate = df.groupby("Cholesterol")["CardioDisease"].mean() * 100
plt.figure(figsize=(6, 4))
bars = plt.bar(["Normal (1)", "Above normal (2)", "Well above (3)"], rate.values,
               color=["#54A24B", "#F58518", "#E45756"])
plt.bar_label(bars, fmt="%.1f%%")
plt.title("Disease Rate by Cholesterol Level")
plt.ylabel("Patients with Disease (%)")
plt.show()

In [ ]:
# Histograms showing the distribution of numeric features
num_cols = ["Age", "SystolicBP", "BMI"]
colors = ["#4C78A8", "#54A24B", "#E45756"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col, c in zip(axes, num_cols, colors):
    ax.hist(df[col], bins=15, color=c, edgecolor="white")
    ax.set_title("Distribution of " + col)
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of the cleaned data
plt.figure(figsize=(9, 6.5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Boxplots: feature values for patients with and without disease
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col, c in zip(axes, ["Age", "SystolicBP", "BMI"], colors):
    sns.boxplot(x="CardioDisease", y=col, data=df, ax=ax, color=c)
    ax.set_title("Boxplot of " + col)
plt.tight_layout()
plt.show()

In [ ]:
# Select features (X) and target (y)
# Height and Weight are dropped because BMI already combines them
X = df.drop(columns=["CardioDisease", "Height", "Weight"])
y = df["CardioDisease"]

# Split the data: 80% training, 20% testing (stratified keeps class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize the features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Features used  :", list(X.columns))
print("Training set   :", X_train.shape)
print("Testing set    :", X_test.shape)

In [ ]:
# Train the Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

print("Model trained successfully")
print("Intercept :", round(model.intercept_[0], 3))
print("Iterations:", model.n_iter_[0])

In [ ]:
# Coefficients and odds ratios of each feature
coef_df = pd.DataFrame({"Feature": X.columns,
                        "Coefficient": model.coef_[0].round(3),
                        "Odds Ratio": np.exp(model.coef_[0]).round(3)})
coef_df = coef_df.sort_values("Coefficient", ascending=False).reset_index(drop=True)
coef_df

In [ ]:
# Bar chart of the coefficients
plt.figure(figsize=(7.5, 4.5))
plot_df = coef_df.sort_values("Coefficient")
plt.barh(plot_df["Feature"], plot_df["Coefficient"],
         color=["#E45756" if c > 0 else "#4C78A8" for c in plot_df["Coefficient"]])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Logistic Regression Coefficients (scaled features)")
plt.xlabel("Coefficient")
plt.show()

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

# Compare the first 8 actual and predicted values
pd.DataFrame({"Actual": y_test.values[:8], "Predicted": y_pred[:8],
              "Probability": y_prob[:8].round(3)})

In [ ]:
# Evaluate the model
acc = accuracy_score(y_test, y_pred)
pre = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cv = cross_val_score(LogisticRegression(max_iter=1000), scaler.transform(X), y, cv=5).mean()

print("Accuracy          :", round(acc, 3))
print("Precision         :", round(pre, 3))
print("Recall            :", round(rec, 3))
print("F1-Score          :", round(f1, 3))
print("ROC-AUC           :", round(auc, 3))
print("5-fold CV Accuracy:", round(cv, 3))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Disease", "Disease"], yticklabels=["No Disease", "Disease"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
print(cm)

In [ ]:
# Classification report
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 4.5))
plt.plot(fpr, tpr, color="#E45756", linewidth=2, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Predict the risk for a new patient
new_patient = pd.DataFrame([[55, 2, 145, 92, 3, 1, 0, 0, 1, 29.0]], columns=X.columns)
new_scaled = scaler.transform(new_patient)
risk = model.predict_proba(new_scaled)[0, 1]
result = model.predict(new_scaled)[0]

print("New patient: Age 55, BP 145/92, Cholesterol 3, BMI 29.0")
print("Risk probability:", round(risk * 100, 1), "%")
print("Predicted class :", result, "(1 = Disease)")